In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)

In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2026-01-01"
sim_start = "2026-05-06"
end = None

In [ ]:
RSI_DIV_CFG = {
    "length": 14,
    "pivot_lookback": 5,
    "min_rsi_delta": 2,

    # divergence filtering
    "os_level": 30,
    "ob_level": 70,
    "zone_mode": "cross",

    # panel styling
    "major_levels": [30, 70],
    "minor_levels": [20, 80],
    "show_zone_shading": True,

    # divergence labels
    "show_div_labels": True,
    "max_div_labels": 6,
    "label_font_size": 10,
    "label_xshift": 10,
    "label_yshift": 14,

    # main chart markers
    "mark_price": True,
    "price_marker_size": 18,
    "marker_offset_mult": 1.35,
}

In [7]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"


### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"


### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"


LONG_EMA_FILTERS = [
    EMA50_1m,
    EMA100_1m,
    EMA150_1m,
    EMA200_1m,
    EMA100_5m,
    EMA200_5m,
    EMA100_15m,
    EMA200_15m
]

In [ ]:
ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]

####### EMA spread features

EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"

EMA100_5m = "5m__ema__EMA_100"
EMA100_15m = "15m__ema__EMA_100"

spread_specs = [
    EmaSpreadSpec(
        name="spread_1m_ema50_vs_1m_ema100",
        left=EMA50_1m,
        right=EMA100_1m,
    ),
    EmaSpreadSpec(
        name="spread_1m_ema50_vs_5m_ema100",
        left=EMA50_1m,
        right=EMA100_5m,
    ),
    EmaSpreadSpec(
        name="spread_1m_ema50_vs_15m_ema100",
        left=EMA50_1m,
        right=EMA100_15m,
    ),
]

####### EMA Diagnostics
LONG_EMA_FILTERS = [
    EMA50_1m,
    EMA100_1m,
    EMA150_1m,
    EMA100_5m,
    EMA200_5m,
    EMA100_15m,
    EMA200_15m
]

from features.ema_diagnostics import (
    EmaPairSpreadSpec,
    EmaGroupSpreadSpec,
    EmaCutThroughSpec,
    add_ema_diagnostics,
)

pair_specs = [
    EmaPairSpreadSpec(
        name="spread_1m_ema50_vs_5m_ema100",
        left=EMA50_1m,
        right=EMA100_5m,
    ),
    EmaPairSpreadSpec(
        name="spread_1m_ema50_vs_15m_ema100",
        left=EMA50_1m,
        right=EMA100_15m,
    ),
]

group_specs = [
    EmaGroupSpreadSpec(
        name="long_ema_filter_spread",
        cols=LONG_EMA_FILTERS,
    )
]

cut_specs = [
    EmaCutThroughSpec(
        name="cut_1m_ema50_through_long_filters",
        leader=EMA50_1m,
        others=[
            EMA100_1m,
            EMA150_1m,
            EMA100_5m,
            EMA200_5m,
            EMA100_15m,
            EMA200_15m,
        ],
        lookbacks=(3, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50),
    )
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
    },
    base_label="1m",
)


In [ ]:
####### EMA spread features (Merged)
merged = add_ema_spread_features(
    merged,
    specs=spread_specs,
    dynamic_windows=(250, 500, 750, 1000, 2000),
    dynamic_quantiles=(0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.9),
)

merged = add_ema_diagnostics(
    merged,
    pair_specs=pair_specs,
    group_specs=group_specs,
    cut_specs=cut_specs,
    dynamic_windows=(250, 500, 750, 1000, 2000),
    dynamic_quantiles=(0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.9),
)

# for col in merged.columns:
#     print(col)

In [ ]:
N_CONFIRM = 1             # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 10
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 50 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


########### CROSS UP MERGED


merged = add_cross_through_features(
    merged,
    specs=[
        CrossThroughSpec(
            name="ema50_1m_cut_up",
            leader=EMA50_1m,
            refs=LONG_EMA_FILTERS,
            direction="up",
            lookback=CROSS_LOOKBACK,
            include_current=True,
            require_current_side=True,
            add_debug_cols=False,
        ),
        
        # This detects zig-zag / failed continuation.
        # require_current_side=False means: count any down-cross event in the lookback,
        # even if EMA50 later moved back above.
        CrossThroughSpec(
            name="ema50_1m_cut_down_any",
            leader=EMA50_1m,
            refs=LONG_EMA_FILTERS,
            direction="down",
            lookback=CROSS_LOOKBACK,
            include_current=True,
            require_current_side=False,
            add_debug_cols=False,
        ),
    ],
)



###################### long rules #########################
open_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)
    ),

    Rule(
        f"Last {N_CONFIRM} closes above all EMA filters",
        lambda c: c.last_n_closes_above_all(LONG_EMA_FILTERS, n=N_CONFIRM)
    ),

    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),

    # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_1m, EMA100_1m)
            and c.gt(EMA100_1m, EMA150_1m)
        )
    ),

        # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_5m, EMA100_5m)
            and c.gt(EMA100_5m, EMA150_5m)
        )
    ),

        # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_15m, EMA100_15m)
            and c.gt(EMA100_15m, EMA150_15m)
        )
    ),

    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, MACD_THRESHOLD_1M)
    ),
    
    Rule(
        "5m MACD above threshold",
        lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
    ),
    
    Rule(
        "15m MACD above threshold",
        lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
    ),

    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_1M, MIN_MACD_DIFF_1M)
    ),
    
    Rule(
        "5m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_5M, MIN_MACD_DIFF_5M)
    ),

    Rule(
        "15m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_15M, MIN_MACD_DIFF_15M)
    ),

    # Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_1m, lookback=1200)
    # ),
    #     Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_15m, lookback=10)
    # ),
    #     Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_15m, lookback=10)
    # ),
)


ema_spread_filter_long_static = ALL(
    Rule(
        f"Last {N_CONFIRM} closes above all EMA filters",
        lambda c: c.last_n_closes_above_all(LONG_EMA_FILTERS, n=N_CONFIRM)
    ),
    Rule(
        f"1m EMA50 vs 5m EMA100 spread above dynamic threshold for {N_SPREAD_CONFIRM} candles",
        lambda c: c.last_all_compare(
            "spread_1m_ema50_vs_5m_ema100__pct",
            ">",
            "spread_1m_ema50_vs_5m_ema100__pct__q50_w500",
            n=N_SPREAD_CONFIRM,
        )
    ),
    
   Rule(
        "EMA basket spread above rolling q80",
        lambda c: c.gt(
            "long_ema_filter_spread__range_pct",
            "long_ema_filter_spread__range_pct_q50_w500",
        )
    ),

    # Rule(
    #     f"1m EMA50 crossed above at least {MIN_EMAS_CROSSED} EMAs in last {CROSS_LOOKBACK} candles",
    #     lambda c: c.crossed_up_through_at_least(
    #         leader=EMA50_1m,
    #         refs=LONG_EMA_FILTERS,
    #         min_crosses=MIN_EMAS_CROSSED,
    #         lookback=CROSS_LOOKBACK,
    #         include_current=True,
    #         require_current_side=True,
    #         unique=True,
    #     )
    # ),

    Rule(
        f"1m EMA50 crossed above at least {MIN_EMAS_CROSSED} EMAs in last {CROSS_LOOKBACK} candles",
        lambda c: c.gte("ema50_1m_cut_up__count", MIN_EMAS_CROSSED)
    ),
    Rule(
        f"No EMA50 downward cuts in last {30} candles",
        lambda c: c.lte("ema50_1m_cut_down_any__count", 0)
    ),
)

    
# )

#EMA75_15m = "15m__ema__EMA_75"

close_rules_long = ALL(
    Rule(
        "Close below EMA100",
        lambda c: c.lt("close", EMA100_15m)
    ),
)

###################### short rules #########################

# open_rules_short = ALL(
#     Rule(
#         f"Close crossed below EMA50 and later {N_CONFIRM} red confirmations",
#         lambda c: c.cross_down_with_later_confirm(
#             left="close",
#             right=EMA50,
#             max_bars_since_cross=MAX_WAIT_AFTER_CROSS,
#             confirm_bars=N_CONFIRM,
#             confirm_pattern="red",
#             require_same_side=True,
#             require_no_opposite_cross=True,
#         )
#     ),
# )


# close_rules_short = ALL(
#     Rule(
#         "Close above EMA50",
#         lambda c: c.gt("close", EMA50)
#     ),
# )

In [ ]:
# ============================================================
# EMA SPREAD / BASKET / CUT-THROUGH OPTIMIZER 
# ============================================================

import time
import itertools
from pathlib import Path
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from simulation.rules import Rule, ALL
from simulation.simulator import Strategy, SimConfig, run_simulation
from features.cross_through import CrossThroughSpec, add_cross_through_features


# ============================================================
# Refined optimizer settings
# ============================================================

DYNAMIC_WINDOWS = (400, 500, 600, 750)
DYNAMIC_QUANTILES = (0.55, 0.60, 0.65, 0.70)

CROSS_LOOKBACKS = (30, 35, 40, 45, 50)
MIN_EMAS_CROSSED_VALUES = (4,)   # important: comma makes this a tuple

N_CONFIRM = 1
N_SPREAD_CONFIRM_VALUES = (2, 3, 4)
N_BASKET_CONFIRM_VALUES = (1,)

INITIAL_CASH = 10_000
MIN_TRADES_FOR_RANKING = 20

RESULTS_PATH = Path("optimization_results_ema_spread_2.csv")
CHECKPOINT_EVERY = 25


# ============================================================
# Columns used by optimizer
# ============================================================

PAIR_SPREAD_COL = "spread_1m_ema50_vs_5m_ema100__pct"
BASKET_SPREAD_COL = "long_ema_filter_spread__range_pct"

EMA50_1m = "1m__ema__EMA_50"

LONG_EMA_CROSS_REFS = [
    "1m__ema__EMA_100",
    "1m__ema__EMA_150",
    "5m__ema__EMA_100",
    "5m__ema__EMA_200",
    "15m__ema__EMA_100",
    "15m__ema__EMA_200",
]


# ============================================================
# Utilities
# ============================================================

def q_int(q: float) -> int:
    return int(round(float(q) * 100))


def pair_q_col(q: float, w: int) -> str:
    # Expected style:
    # spread_1m_ema50_vs_5m_ema100__pct__q60_w500
    return f"{PAIR_SPREAD_COL}__q{q_int(q)}_w{int(w)}"


def basket_q_col(q: float, w: int) -> str:
    # Expected style:
    # long_ema_filter_spread__range_pct_q50_w500
    return f"{BASKET_SPREAD_COL}_q{q_int(q)}_w{int(w)}"


def rolling_all_bool(mask, n: int) -> np.ndarray:
    """
    True when mask has been true for the last n candles including current.
    """
    mask = pd.Series(mask).fillna(False).astype(bool)

    if int(n) <= 1:
        return mask.to_numpy()

    return mask.rolling(int(n), min_periods=int(n)).sum().eq(int(n)).to_numpy()


def ensure_dynamic_threshold(df: pd.DataFrame, base_col: str, threshold_col: str, q: float, w: int) -> None:
    """
    Adds rolling quantile threshold if missing.
    Uses shift(1) to avoid lookahead.
    """
    if base_col not in df.columns:
        raise KeyError(f"Missing base column: {base_col}")

    if threshold_col in df.columns:
        return

    df[threshold_col] = (
        df[base_col]
        .rolling(int(w), min_periods=max(50, int(w * 0.20)))
        .quantile(float(q))
        .shift(1)
    )


def _to_dataframe(obj):
    """
    Handles DataFrame, list[dict], list[dataclass], list[object], empty list, etc.
    """
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()

        rows = []
        for x in obj:
            if isinstance(x, dict):
                rows.append(x)
            elif is_dataclass(x):
                rows.append(asdict(x))
            elif hasattr(x, "__dict__"):
                rows.append(vars(x))
            else:
                rows.append(x)

        return pd.DataFrame(rows)

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    return pd.DataFrame(obj)


def make_sim_config():
    """
    Tries the quiet SimConfig first. Falls back if your local SimConfig
    version does not support log/progress args.
    """
    kwargs = dict(
        initial_cash=INITIAL_CASH,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        sim_start="2026-01-01",
        sim_end="2026-05-18",
        sim_tz=tz,
    )

    try:
        return SimConfig(
            **kwargs,
            log_level="WARNING",
            progress=False,
            progress_bar=False,
        )
    except TypeError:
        return SimConfig(**kwargs)


def summarize_sim_result(res, params: dict) -> dict:
    trades = _to_dataframe(getattr(res, "trades", None))

    if trades.empty or "pnl" not in trades.columns:
        closed = pd.DataFrame()
    else:
        closed = trades[trades["pnl"].notna()].copy()

    num_trades = len(closed)

    if num_trades:
        pnl = float(closed["pnl"].sum())
        wins = closed[closed["pnl"] > 0]
        losses = closed[closed["pnl"] <= 0]

        win_rate = float(len(wins) / num_trades * 100)
        avg_pnl = float(closed["pnl"].mean())
        avg_winner = float(wins["pnl"].mean()) if len(wins) else np.nan
        avg_loser = float(losses["pnl"].mean()) if len(losses) else np.nan
        best_trade = float(closed["pnl"].max())
        worst_trade = float(closed["pnl"].min())
    else:
        pnl = 0.0
        win_rate = 0.0
        avg_pnl = 0.0
        avg_winner = np.nan
        avg_loser = np.nan
        best_trade = np.nan
        worst_trade = np.nan

    final_equity = INITIAL_CASH + pnl

    equity_curve = _to_dataframe(getattr(res, "equity_curve", None))
    if not equity_curve.empty:
        for col in ["equity", "balance", "cash"]:
            if col in equity_curve.columns:
                final_equity = float(equity_curve[col].iloc[-1])
                break

    out = {
        **params,
        "num_trades": num_trades,
        "win_rate": win_rate,
        "pnl": pnl,
        "final_equity": final_equity,
        "return_pct": (final_equity / INITIAL_CASH - 1) * 100,
        "avg_pnl": avg_pnl,
        "avg_winner": avg_winner,
        "avg_loser": avg_loser,
        "best_trade": best_trade,
        "worst_trade": worst_trade,
    }

    stats = getattr(res, "stats", {})
    if isinstance(stats, pd.Series):
        stats = stats.to_dict()

    if isinstance(stats, dict):
        for k in [
            "max_drawdown",
            "max_drawdown_pct",
            "profit_factor",
            "total_return",
            "total_return_pct",
        ]:
            if k in stats:
                out[k] = stats[k]

    return out


# ============================================================
# Validate required columns
# ============================================================

required_cols = [
    "open",
    "close",
    PAIR_SPREAD_COL,
    BASKET_SPREAD_COL,
    EMA50_1m,
    *LONG_EMA_FILTERS,
    *LONG_EMA_CROSS_REFS,
]

missing = [c for c in required_cols if c not in merged.columns]
if missing:
    raise KeyError(f"Missing required columns in merged: {missing}")


# ============================================================
# Prepare dynamic threshold columns
# ============================================================

print("Preparing dynamic threshold columns...")

for w in DYNAMIC_WINDOWS:
    for q in DYNAMIC_QUANTILES:
        ensure_dynamic_threshold(
            merged,
            base_col=PAIR_SPREAD_COL,
            threshold_col=pair_q_col(q, w),
            q=q,
            w=w,
        )

        ensure_dynamic_threshold(
            merged,
            base_col=BASKET_SPREAD_COL,
            threshold_col=basket_q_col(q, w),
            q=q,
            w=w,
        )


# ============================================================
# Prepare cut-through count columns
# ============================================================

print("Preparing cut-through count columns...")

cross_specs = [
    CrossThroughSpec(
        name=f"ema50_1m_cut_up_lb{lb}",
        leader=EMA50_1m,
        refs=LONG_EMA_CROSS_REFS,
        direction="up",
        lookback=lb,
        include_current=True,
        require_current_side=True,
        add_debug_cols=False,
    )
    for lb in CROSS_LOOKBACKS
]

merged = add_cross_through_features(merged, specs=cross_specs)


# ============================================================
# Precompute base confirmation signal
# ============================================================

print("Preparing base confirmation signal...")

above_all_emas = np.logical_and.reduce([
    merged["close"].to_numpy(dtype=float) > merged[c].to_numpy(dtype=float)
    for c in LONG_EMA_FILTERS
])

green_candle = merged["close"].to_numpy(dtype=float) > merged["open"].to_numpy(dtype=float)

BASE_CLOSE_CONFIRM = rolling_all_bool(above_all_emas, N_CONFIRM)
BASE_GREEN_CONFIRM = rolling_all_bool(green_candle, N_CONFIRM)

BASE_SIGNAL = BASE_CLOSE_CONFIRM & BASE_GREEN_CONFIRM


# ============================================================
# Resume support
# ============================================================

if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done_keys = set(previous["key"].astype(str))
    results = previous.to_dict("records")
    print(f"Resuming | completed={len(done_keys):,}")
else:
    done_keys = set()
    results = []


# ============================================================
# Build grid
# ============================================================

grid = list(itertools.product(
    DYNAMIC_WINDOWS,              # spread_window
    DYNAMIC_QUANTILES,            # spread_quantile
    DYNAMIC_WINDOWS,              # basket_window
    DYNAMIC_QUANTILES,            # basket_quantile
    CROSS_LOOKBACKS,
    MIN_EMAS_CROSSED_VALUES,
    N_SPREAD_CONFIRM_VALUES,
    N_BASKET_CONFIRM_VALUES,
))

print(f"Total combinations: {len(grid):,}")
print(f"Already completed: {len(done_keys):,}")
print(f"Remaining: {len(grid) - len(done_keys):,}")


# ============================================================
# Run optimizer
# ============================================================

t0 = time.time()
new_runs = 0

for (
    spread_window,
    spread_q,
    basket_window,
    basket_q,
    cross_lookback,
    min_emas_crossed,
    n_spread_confirm,
    n_basket_confirm,
) in tqdm(grid, desc="Optimizing"):

    key = (
        f"spread_w{spread_window}_q{q_int(spread_q)}__"
        f"basket_w{basket_window}_q{q_int(basket_q)}__"
        f"cross_lb{cross_lookback}_min{min_emas_crossed}__"
        f"spconf{n_spread_confirm}_bconf{n_basket_confirm}"
    )

    if key in done_keys:
        continue

    spread_threshold_col = pair_q_col(spread_q, spread_window)
    basket_threshold_col = basket_q_col(basket_q, basket_window)
    cut_count_col = f"ema50_1m_cut_up_lb{cross_lookback}__count"

    spread_signal = rolling_all_bool(
        merged[PAIR_SPREAD_COL].to_numpy(dtype=float) >
        merged[spread_threshold_col].to_numpy(dtype=float),
        n_spread_confirm,
    )

    basket_signal = rolling_all_bool(
        merged[BASKET_SPREAD_COL].to_numpy(dtype=float) >
        merged[basket_threshold_col].to_numpy(dtype=float),
        n_basket_confirm,
    )

    cut_signal = merged[cut_count_col].to_numpy(dtype=float) >= float(min_emas_crossed)

    merged["__opt_open_signal"] = BASE_SIGNAL & spread_signal & basket_signal & cut_signal

    open_rules_long = ALL(
        Rule(
            key,
            lambda c: c.flag("__opt_open_signal")
        )
    )

    strategy = Strategy(
        open_rules_long=open_rules_long,
        close_rules_long=close_rules_long,
    )

    res = run_simulation(
        df=merged,
        strategy=strategy,
        cfg=make_sim_config(),
        time_col="t",
        price_col="close",
    )

    params = {
        "key": key,
        "spread_window": spread_window,
        "spread_quantile": spread_q,
        "basket_window": basket_window,
        "basket_quantile": basket_q,
        "cross_lookback": cross_lookback,
        "min_emas_crossed": min_emas_crossed,
        "n_confirm": N_CONFIRM,
        "n_spread_confirm": n_spread_confirm,
        "n_basket_confirm": n_basket_confirm,
    }

    results.append(summarize_sim_result(res, params))
    done_keys.add(key)
    new_runs += 1

    if new_runs % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
        elapsed_min = (time.time() - t0) / 60
        print(
            f"Checkpoint saved | new_runs={new_runs:,} | "
            f"total_done={len(done_keys):,} | elapsed={elapsed_min:.1f} min"
        )

# Final save
pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print(f"Done. Saved: {RESULTS_PATH}")


# ============================================================
# Display best results
# ============================================================

opt = pd.read_csv(RESULTS_PATH)

valid = opt[opt["num_trades"] >= MIN_TRADES_FOR_RANKING].copy()

best_by_profit = valid.sort_values(
    ["pnl", "final_equity", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_winrate = valid.sort_values(
    ["win_rate", "pnl", "num_trades"],
    ascending=False,
)

best_by_profit_factor = valid.sort_values(
    ["profit_factor", "pnl", "num_trades"],
    ascending=False,
) if "profit_factor" in valid.columns else pd.DataFrame()

print("\nBEST BY PROFIT")
display(best_by_profit.head(20))

print("\nBEST BY WIN RATE")
display(best_by_winrate.head(20))

if not best_by_profit_factor.empty:
    print("\nBEST BY PROFIT FACTOR")
    display(best_by_profit_factor.head(20))

In [ ]:
strategy = Strategy(
    open_rules_long=ema_spread_filter_long_static,
    close_rules_long=close_rules_long,

    # allow_short=True,
    # open_rules_short=open_rules_short,
    # close_rules_short=close_rules_short,
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(
        initial_cash=10_000,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        # sim_start=sim_start,
        sim_start="2026-01-01",
        sim_end="2026-05-18",
        #sim_start="2026-05-10",
        #sim_end="2026-05-15",
        sim_tz=tz,
    ),
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
ema_pair_panel = ema_pair_spread_panel(
    name="spread_1m_ema50_vs_5m_ema100",
    title="Spread: 1m EMA50 vs 5m EMA100 (%)",
    mode="pct",
    dynamic_threshold_cols=[
        "spread_1m_ema50_vs_5m_ema100__pct__q50_w500",
    ],
)

ema_group_panel = ema_group_spread_panel(
    name="long_ema_filter_spread",
    title="EMA Basket Spread / Compression",
    metrics=("range_pct", "std_pct", "mad_pct"),
    dynamic_threshold_cols=[
        "long_ema_filter_spread__range_pct_q50_w500",
    ],
)

ema_cut_panel = ema_cut_through_panel(
    name="cut_1m_ema50_through_long_filters",
    lookback=CROSS_LOOKBACK,
    title="1m EMA50 Cut-Through Speed",
)

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
# plot_indicators = make_plot_indicators(
#     base_specs=ind_1m,
#     mtf_specs={
#         "5m": ind_5m,
#         "15m": ind_15m,
#     },
#     mtf_include={
#         "5m": ["macd_8_21_5"],
#         "15m": ["macd_8_21_5"],
#     }
# )

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
}

plot_toggles = [
    PlotToggle("1m", "ema", visible=True),
    PlotToggle("5m", "ema", visible=True),
    PlotToggle("15m", "ema", visible=True),


    # Available in legend, hidden by default
    PlotToggle("1m", "macd_8_21_5", visible=False),
    PlotToggle("5m", "macd_8_21_5", visible=False),
    PlotToggle("15m", "macd_8_21_5", visible=False),

    # Not plotted at all, but still can be used in rules
    PlotToggle("1m", "rsi14", visible=False),
    PlotToggle("5m", "rsi14", visible=False),
    PlotToggle("15m", "rsi14", visible=False),
]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

plot_indicators = plot_indicators + [
    ema_pair_panel,
    ema_group_panel,
    ema_cut_panel,   # optional
]

assert_plot_columns_exist(merged, plot_indicators)

In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-05-06",
        # date_to="2026-05-10",
        date_from="2026-1-07",
        date_to="2026-01-14",
        height=1400,
        width=1900,
    ),
)

# plot_simulation(
#     df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
#     symbol=symbol,
#     interval="5m",
#     market=market,
#     trades=res.trades,
#     ma_windows=[20],
#     indicators=ind_5m,
#     plot_cfg=PlotConfig(
#         tz="Asia/Karachi",
#         date_from="2026-02-01",
#         date_to="2026-02-05",
#         height=1400,
#         width=1900
#     ),
# )